In [6]:
import numpy as np

In [11]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

In [12]:
x = [2.0, 3.0]

In [9]:
# initial states
h_prev = 0.0
c_prev = 0.0
# weights (start simple — all 1.0 like your RNN)
w_f, w_xf = 1.0, 1.0   # forget gate
w_i, w_xi = 1.0, 1.0   # input gate
w_c, w_xc = 1.0, 1.0   # candidate cell
w_o, w_xo = 1.0, 1.0   # output gate
w_y = 2.0              # output projection (like Wy in RNN)
h_states = [h_prev]
c_states = [c_prev]

In [13]:
for t in range(len(x)):
    h_old = h_states[-1] #Hidden state
    c_old = c_states[-1] #Memory

    f = sigmoid(w_f * h_old + w_xf * x[t]) #Forget gate
    i = sigmoid(w_i * h_old + w_xi * x[t]) #Input gate
    c_candidate = np.tanh(w_c * h_old + w_xc * x[t]) #Candidate cell(what to add to memory)
    o = sigmoid(w_o * h_old + w_xo * x[t]) #Output gate

    c_new = f * c_old + i * c_candidate # What to keep from old memory + what to add from new candidate
    h_new = o * np.tanh(c_new) # Output of LSTM

    h_states.append(h_new) #Store the new hidden state
    c_states.append(c_new) #Store the new memory

y = w_y * h_states[-1] #Output of LSTM
print("h_states:", h_states)
print("c_states:", c_states)
print("y:", y)

h_states: [0.0, np.float64(0.6082834181835157), np.float64(0.9217148448789616)]
c_states: [0.0, np.float64(0.8491126756208685), np.float64(1.798897995274348)]
y: 1.8434296897579232


In [ ]:
# Step 2: Forward + Manual Backward Pass (like RNN Cell 2)

x = np.array([1.0, 2.0, 3.0])
target = 0.0

# Weights
w_f, w_xf = 1.0, 1.0
w_i, w_xi = 1.0, 1.0
w_c, w_xc = 1.0, 1.0
w_o, w_xo = 1.0, 1.0
w_y = 1.0

# --- Forward ---
h_states = [0.0]
c_states = [0.0]
f_gates = []
i_gates = []
c_candidates = []
o_gates = []

for t in range(len(x)):
    h_old = h_states[-1]
    c_old = c_states[-1]

    f = sigmoid(w_f * h_old + w_xf * x[t])
    i = sigmoid(w_i * h_old + w_xi * x[t])
    c_candidate = np.tanh(w_c * h_old + w_xc * x[t])
    o = sigmoid(w_o * h_old + w_xo * x[t])

    c_new = f * c_old + i * c_candidate
    h_new = o * np.tanh(c_new)

    f_gates.append(f)
    i_gates.append(i)
    c_candidates.append(c_candidate)
    o_gates.append(o)
    h_states.append(h_new)
    c_states.append(c_new)

y = w_y * h_states[-1]
loss = 0.5 * (y - target) ** 2

print("Hidden states:", h_states)
print("Cell states:", c_states)
print("Prediction:", y)
print("Loss:", loss)

# --- Backward ---
dw_f, dw_xf = 0.0, 0.0
dw_i, dw_xi = 0.0, 0.0
dw_c, dw_xc = 0.0, 0.0
dw_o, dw_xo = 0.0, 0.0
dw_y = 0.0

# Loss -> output
dy = y - target
dw_y = dy * h_states[-1]

# Gradient entering final hidden state
dh = dy * w_y
dc = 0.0  # gradient flowing into cell state from future timesteps

# Back through time
for t in reversed(range(len(x))):
    h_old = h_states[t]
    c_old = c_states[t]
    c_new = c_states[t + 1]
    f = f_gates[t]
    i = i_gates[t]
    c_candidate = c_candidates[t]
    o = o_gates[t]
    tanh_c = np.tanh(c_new)

    # Gradient into cell state: from future + through h = o * tanh(c)
    dc_total = dc + dh * o * (1 - tanh_c ** 2)

    # Through c_new = f * c_old + i * c_candidate
    df = dc_total * c_old
    di = dc_total * c_candidate
    dc_candidate = dc_total * i

    # Through sigmoid gates (derivative: gate * (1 - gate))
    dz_f = df * f * (1 - f)
    dz_i = di * i * (1 - i)
    dz_c = dc_candidate * (1 - c_candidate ** 2)  # through tanh
    dz_o = (dh * tanh_c) * o * (1 - o)

    # Accumulate weight gradients
    dw_f += dz_f * h_old
    dw_xf += dz_f * x[t]
    dw_i += dz_i * h_old
    dw_xi += dz_i * x[t]
    dw_c += dz_c * h_old
    dw_xc += dz_c * x[t]
    dw_o += dz_o * h_old
    dw_xo += dz_o * x[t]

    # Send gradient to previous hidden state (all 4 gates used h_old)
    dh = dz_f * w_f + dz_i * w_i + dz_c * w_c + dz_o * w_o

    # Send gradient to previous cell state
    dc = dc_total * f

# Update weights
learning_rate = 0.01
w_f -= learning_rate * dw_f
w_xf -= learning_rate * dw_xf
w_i -= learning_rate * dw_i
w_xi -= learning_rate * dw_xi
w_c -= learning_rate * dw_c
w_xc -= learning_rate * dw_xc
w_o -= learning_rate * dw_o
w_xo -= learning_rate * dw_xo
w_y -= learning_rate * dw_y

print("\nGradients:")
print("dw_f =", dw_f, " dw_xf =", dw_xf)
print("dw_i =", dw_i, " dw_xi =", dw_xi)
print("dw_c =", dw_c, " dw_xc =", dw_xc)
print("dw_o =", dw_o, " dw_xo =", dw_xo)
print("dw_y =", dw_y)

print("\nUpdated weights:")
print("w_f =", w_f, " w_xf =", w_xf)
print("w_i =", w_i, " w_xi =", w_xi)
print("w_c =", w_c, " w_xc =", w_xc)
print("w_o =", w_o, " w_xo =", w_xo)
print("w_y =", w_y)

In [15]:
# Step 3: Training Loop with WhyyTorch autograd (like RNN Cell 3)

import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "00-foundation"))

from Autograd import WhyyTorch as wt


def wt_sigmoid(z):
    one = wt(1.0, requires_grad=False)
    return one / (one + (-z).exp())


# Data
x_data = [1.0, 2.0, 3.0]
target = wt(100.0, requires_grad=False)

# Weights
w_f, w_xf = wt(1.0), wt(1.0)
w_i, w_xi = wt(1.0), wt(1.0)
w_c, w_xc = wt(1.0), wt(1.0)
w_o, w_xo = wt(1.0), wt(1.0)
w_y = wt(1.0)

weights = [w_f, w_xf, w_i, w_xi, w_c, w_xc, w_o, w_xo, w_y]
lr = 0.1
epochs = 10

for epoch in range(epochs):
    for w in weights:
        w.zero_grad()

    h_states = [wt(0.0, requires_grad=False)]
    c_states = [wt(0.0, requires_grad=False)]

    # Forward pass
    for t in range(len(x_data)):
        xi = wt(x_data[t], requires_grad=False)
        h_old = h_states[-1]
        c_old = c_states[-1]

        f = wt_sigmoid(w_f * h_old + w_xf * xi)
        i_gate = wt_sigmoid(w_i * h_old + w_xi * xi)
        c_candidate = (w_c * h_old + w_xc * xi).tanh()
        o = wt_sigmoid(w_o * h_old + w_xo * xi)

        c_new = f * c_old + i_gate * c_candidate
        h_new = o * c_new.tanh()

        h_states.append(h_new)
        c_states.append(c_new)

    # Output + loss
    y = w_y * h_states[-1]
    loss = 0.5 * (y - target) ** 2

    # Backward pass
    loss.backward()

    # Update weights
    for w in weights:
        w.data -= lr * w.grad

    print(
        f"epoch {epoch}: loss={loss.data}, "
        f"w_y={w_y.data}, w_f={w_f.data}, w_o={w_o.data}"
    )

epoch 0: loss=4904.373046875, w_y=10.516509056091309, w_f=1.0143979787826538, w_o=1.1728228330612183
epoch 1: loss=4019.337646484375, w_y=19.333036422729492, w_f=1.1213328838348389, w_o=1.3774657249450684
epoch 2: loss=3265.4892578125, w_y=27.352825164794922, w_f=1.1675916910171509, w_o=1.4098012447357178
epoch 3: loss=2652.399658203125, w_y=34.58647155761719, w_f=1.2061768770217896, w_o=1.4387413263320923
epoch 4: loss=2153.861572265625, w_y=41.10810089111328, w_f=1.237669587135315, w_o=1.4628527164459229
epoch 5: loss=1748.8173828125, w_y=46.98637390136719, w_f=1.2636579275131226, w_o=1.4827821254730225
epoch 6: loss=1419.852294921875, w_y=52.28404235839844, w_f=1.285390019416809, w_o=1.4993658065795898
epoch 7: loss=1152.7249755859375, w_y=57.058082580566406, w_f=1.303771734237671, w_o=1.5132964849472046
epoch 8: loss=935.83154296875, w_y=61.36003112792969, w_f=1.3194663524627686, w_o=1.5251060724258423
epoch 9: loss=759.7366943359375, w_y=65.23644256591797, w_f=1.332970380783081, w